# 02b - Fetch and Merge County Health Rankings Smoking Data

This notebook extracts county-level adult smoking rates from County Health Rankings (CHR) analytic data files for 2012-2019 and prepares them for integration into the lung cancer prediction pipeline.

Smoking is the dominant risk factor for lung cancer. Including it as a predictor allows the model to isolate independent atmospheric contributions. A dummy variable (`is_post_2015`) is added to account for the CHR methodology change from BRFSS moving averages (2012-2015) to MRP-based estimates (2016-2019).

In [5]:
import pandas as pd
import os

## 1. Processing Function

Each CHR CSV has a metadata row (row 1) containing variable codes. We skip it with `skiprows=[1]`. State and national summary rows are filtered out using `County FIPS Code != 0`.

In [6]:
def process_smoking_data(year):
    """
    Extracts county-level adult smoking rates from a CHR CSV file.

    Parameters:
    year (int): The CHR release year (2012-2019).

    Returns:
    pd.DataFrame: DataFrame with columns [fips, Year, smoking_rate].
    """
    path = f'../data/raw/chr_smoking/chr_{year}.csv'
    df = pd.read_csv(path, skiprows=[1],
                     usecols=['5-digit FIPS Code', 'County FIPS Code', 'Adult smoking raw value'],
                     low_memory=False)
    print(f"Loaded {len(df)} rows for {year}")

    # Filter to county rows only
    df = df[df['County FIPS Code'] != 0]
    print(f"  After removing state/national rows: {len(df)}")

    # Drop rows with missing FIPS or smoking values
    df = df.dropna(subset=['5-digit FIPS Code', 'Adult smoking raw value'])
    print(f"  After dropping missing values: {len(df)}")

    # Zero-pad FIPS to 5 digits
    df['fips'] = df['5-digit FIPS Code'].astype(int).astype(str).str.zfill(5)
    df = df.rename(columns={'Adult smoking raw value': 'smoking_rate'})
    df['Year'] = year

    return df[['fips', 'Year', 'smoking_rate']]

## 2. Test with a Single Year

In [7]:
df_test = process_smoking_data(2012)
print(f"\nShape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")
print(f"\nSmoking Rate Statistics (2012):")
print(df_test['smoking_rate'].describe())
df_test.head()

Loaded 3193 rows for 2012
  After removing state/national rows: 3141
  After dropping missing values: 2506

Shape: (2506, 3)
Columns: ['fips', 'Year', 'smoking_rate']

Smoking Rate Statistics (2012):
count    2506.000000
mean        0.211565
std         0.058894
min         0.035000
25%         0.174000
50%         0.209000
75%         0.249000
max         0.482000
Name: smoking_rate, dtype: float64


,fips,Year,smoking_rate
2,01001,2012,0.246
3,01003,2012,0.227
4,01005,2012,0.234
5,01007,2012,0.351
6,01009,2012,0.227


## 3. Process All Years (2012-2019)

In [8]:
all_years = []
for year in range(2012, 2020):
    df_year = process_smoking_data(year)
    all_years.append(df_year)
    print("---")

smoking_df = pd.concat(all_years, ignore_index=True)

# Add dummy variable for CHR methodology break
smoking_df['is_post_2015'] = (smoking_df['Year'] >= 2016).astype(int)

print(f"\nCombined dataset shape: {smoking_df.shape}")
print(f"Columns: {smoking_df.columns.tolist()}")

Loaded 3193 rows for 2012
  After removing state/national rows: 3141
  After dropping missing values: 2506
---
Loaded 3193 rows for 2013
  After removing state/national rows: 3141
  After dropping missing values: 2536
---
Loaded 3193 rows for 2014
  After removing state/national rows: 3141
  After dropping missing values: 2711
---
Loaded 3193 rows for 2015
  After removing state/national rows: 3141
  After dropping missing values: 2711
---
Loaded 3193 rows for 2016
  After removing state/national rows: 3141
  After dropping missing values: 3140
---
Loaded 3195 rows for 2017
  After removing state/national rows: 3143
  After dropping missing values: 3135
---
Loaded 3194 rows for 2018
  After removing state/national rows: 3142
  After dropping missing values: 3142
---
Loaded 3194 rows for 2019
  After removing state/national rows: 3142
  After dropping missing values: 3142
---

Combined dataset shape: (23023, 4)
Columns: ['fips', 'Year', 'smoking_rate', 'is_post_2015']


## 4. Validation

In [9]:
# County coverage per year
print("County coverage per year:")
print(smoking_df.groupby('Year')['fips'].count())

print("\nSummary statistics by year:")
print(smoking_df.groupby('Year')['smoking_rate'].describe())

print(f"\nMissing values:\n{smoking_df.isnull().sum()}")

# Confirm known CHR behavior: 2014==2015 and 2018==2019
for y1, y2 in [(2014, 2015), (2018, 2019)]:
    df_a = smoking_df[smoking_df['Year'] == y1].set_index('fips')['smoking_rate']
    df_b = smoking_df[smoking_df['Year'] == y2].set_index('fips')['smoking_rate']
    common = df_a.index.intersection(df_b.index)
    identical = (df_a.loc[common] == df_b.loc[common]).all()
    print(f"\n{y1} vs {y2}: {len(common)} overlapping counties, values identical = {identical}")

County coverage per year:
Year
2012    2506
2013    2536
2014    2711
2015    2711
2016    3140
2017    3135
2018    3142
2019    3142
Name: fips, dtype: int64

Summary statistics by year:
       count      mean       std       min       25%       50%       75%  \
Year                                                                       
2012  2506.0  0.211565  0.058894  0.035000  0.174000  0.209000  0.249000   
2013  2536.0  0.204667  0.060883  0.000000  0.165000  0.204000  0.242000   
2014  2711.0  0.212809  0.063140  0.031000  0.170000  0.208000  0.249000   
2015  2711.0  0.212809  0.063140  0.031000  0.170000  0.208000  0.249000   
2016  3140.0  0.183989  0.037890  0.069000  0.157000  0.178000  0.207000   
2017  3135.0  0.178844  0.035380  0.065462  0.154076  0.173382  0.199751   
2018  3142.0  0.178726  0.036601  0.067354  0.152351  0.173209  0.202803   
2019  3142.0  0.178726  0.036601  0.067354  0.152351  0.173209  0.202803   

           max  
Year            
2012  0.482000  

## 5. Save Output

In [10]:
output_dir = '../data/processed/smoking_by_year/'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'smoking_all_years.csv')
smoking_df.to_csv(output_path, index=False)
print(f"Saved {len(smoking_df)} rows to {output_path}")

Saved 23023 rows to ../data/processed/smoking_by_year/smoking_all_years.csv


## 6. Verify Output File

In [11]:
df_check = pd.read_csv(output_path)
print(f"Output file: {output_path}")
print(f"Shape: {df_check.shape}")
print(f"Columns: {df_check.columns.tolist()}")
print(f"\nFIPS sample: {df_check['fips'].head().tolist()}")
df_check.head(10)

Output file: ../data/processed/smoking_by_year/smoking_all_years.csv
Shape: (23023, 4)
Columns: ['fips', 'Year', 'smoking_rate', 'is_post_2015']

FIPS sample: [1001, 1003, 1005, 1007, 1009]


,fips,Year,smoking_rate,is_post_2015
0,1001,2012,0.246,0
1,1003,2012,0.227,0
2,1005,2012,0.234,0
3,1007,2012,0.351,0
4,1009,2012,0.227,0
5,1013,2012,0.287,0
6,1015,2012,0.268,0
7,1017,2012,0.214,0
8,1019,2012,0.252,0
9,1021,2012,0.226,0
